In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# ENDING VOLUME — LADDER BACKTEST ENGINE
# ═══════════════════════════════════════════════════════════════════════
#
# This notebook simulates the COMPLETE strategy, bar by bar, including
# the take-profit ladder that the penetration study could not model.
#
# Everything before this notebook measured a stripped-down version:
# full size held to VWAP or the brake, nothing in between. That is two
# of the five exit rules. This one runs all of them.
#
#
# ═══════════════════════════════════════════════════════════════════════
# PART 1 — HOW A TRADE IS FOUND
# ═══════════════════════════════════════════════════════════════════════
#
# SAME DAY ONLY. No multi-day levels. Two timeframes run in parallel:
# 2-minute is execution, 5-minute is the fallback for when volume
# disperses across two 2-min bars so neither clears alone.
#
# COMPARISON POOL starts at 9:30 and includes the opening-range candles.
#
#   RANK PATH   the candle becomes the new #1 or #2 by volume in the
#               session. May only FIRE after the opening range:
#               9:46 on the 2-min, 9:45 on the 5-min.
#
#   2X PATH     the candle prints >= 2x the session's REIGNING highest
#               volume candle. The benchmark updates each time a new
#               highest prints, so every 2x makes the next one twice as
#               hard — the path self-damps. Eligible from 9:32 (2m) /
#               9:35 (5m), so it can fire inside the opening range.
#
# DIRECTION IS ALWAYS COUNTER-TREND.
#
#   spike BELOW the 15-min opening range low   ->  LONG
#   spike ABOVE the 15-min opening range high  ->  SHORT
#   inside the range                           ->  no trade
#
#   Pre-9:45 (2x path only, no completed range yet): low of day is
#   LONG, high of day is SHORT, otherwise compare to session open.
#
# NO INTRABAR ENTRIES. The signal candle must CLOSE. Entry is the
# NEXT bar's open. This is what makes the stop, R, and the ladder
# gate all knowable before you are in the position.
#
#
# ═══════════════════════════════════════════════════════════════════════
# PART 2 — THE TWO STOPS
# ═══════════════════════════════════════════════════════════════════════
#
#   SOFT STOP   at the signal candle's low (long) / high (short).
#               Evaluated on CLOSES ONLY. Ignores intrabar pokes.
#               Exists because a climax is usually not one bar —
#               MSFU at $39 and the NVDL 10:00/10:05 sequence both had
#               the second bar of the same climax poke through the
#               first bar's extreme and then work.
#
#   HARD BRAKE  further out, fires INTRABAR, always. Placed at
#               BRAKE_FRAC x the signal candle's own range beyond the
#               soft stop. NOT a multiple of R and NOT fixed cents:
#               a candle closing on its low makes R about a penny, and
#               a 1.5R brake would then sit inside normal noise; a
#               fixed 2c is 0.4% on a $5 stock and 0.05% on a $36 one.
#
#   R IS MEASURED ENTRY TO BRAKE, never entry to soft stop. The brake
#   is the real maximum loss, so it governs sizing, the Case 1 / Case 2
#   gate, and every R-denominated number in the output.
#
#   TWO SANITY GATES sit alongside the floor, both learned from
#   replaying the playbook trades:
#
#     entry_past_soft_stop  the bar we enter on opened BEYOND the signal
#                           candle's extreme, so we would be entering on
#                           the wrong side of our own stop. R collapses
#                           to whatever slice of the brake buffer is
#                           left and a 1R loss costs several R.
#
#     target_too_close      VWAP is nearer than MIN_TARGET_R. There is
#                           nothing to aim at; the round trip costs more
#                           than the move is worth.
#
#   THE AFFORDABILITY GATE. Friction is a fixed number of cents per
#   share. What decides whether a trade can carry it is NOT the size of
#   the risk unit but the size of the MOVE BEING AIMED FOR:
#
#       target 35c, round trip 1.5c  ->  4% of the move   TAKE IT
#       target  6c, round trip 1.5c  -> 25% of the move   SKIP IT
#
#   Measured on 798k signals this beats a floor on R by a wide margin:
#   cost <= 5% of target scores +0.165R per trade against +0.119R for
#   the best fixed-R floor, on MORE trades. And at ZERO cost a floor on
#   R strictly hurts (+0.202 unfiltered vs +0.163 at a 15c floor) —
#   small-R trades are good trades that simply cannot pay the toll.
#   NVDL 2026-08-26 is the example: a 3.7c risk unit, a 35c target, and
#   +7.7R. Any floor on R would have thrown it away.
#
#   MIN_R_DOLLARS still exists as a blunt secondary backstop but is OFF
#   by default. It is NOT a minimum stop. The stop goes where the
#   candle says it goes. The floor decides which signals are worth
#   TAKING at all.
#
#   The reason is friction, not risk. Round-trip cost is a fixed number
#   of cents per share; R is whatever the candle gives you. So cost as a
#   fraction of your risk unit is entirely determined by how wide the
#   candle was:
#
#       R = 3c,  cost 1.5c round trip  ->  50% of R gone to friction
#       R = 15c, cost 1.5c round trip  ->  10% of R gone
#       R = 60c, cost 1.5c round trip  ->  2.5% of R gone
#
#   No exit rule repairs the first line. The ladder cannot help a trade
#   that hands half its risk unit to the spread before it starts.
#
#   So the floor is expressed as a MULTIPLE OF THE ROUND TRIP
#   (MIN_R_COST_MULT), not as a fixed dollar amount. It moves with your
#   actual fills: better execution lowers the bar, worse raises it.
#
#
# ═══════════════════════════════════════════════════════════════════════
# PART 3 — EXIT RULES  (the part this notebook exists to test)
# ═══════════════════════════════════════════════════════════════════════
#
# THE BRANCH IS CHOSEN AT ENTRY and never re-evaluated:
#
#     distance from entry to VWAP  >=  1R   ->   CASE 1  (ladder ON)
#     distance from entry to VWAP  <   1R   ->   CASE 2  (ladder OFF)
#
# Why the branch exists: the ending-volume candle that creates the
# signal also DRAGS VWAP TOWARD THE ENTRY, because huge volume printing
# at an extreme price pulls the average toward that price. So the
# biggest signals produce the tightest targets. When VWAP is closer than
# 1R, laddering out would bank three partial exits inside a band
# narrower than the risk taken — worse than simply waiting.
#
# ─────────────────────────────────────────────────────────────────────
# CASE 1 — VWAP is at least 1R away. Ladder active.
# Percentages are of the ORIGINAL position, not of what remains.
# ─────────────────────────────────────────────────────────────────────
#
#   VOLUME EVENT 1, in profit, before VWAP   ->  sell 50%
#   VOLUME EVENT 2, in profit, before VWAP   ->  sell 25%
#   VWAP REACHED                             ->  top up to 90% sold,
#                                                stop to BREAK-EVEN
#   remaining 10%                            ->  trails, ratchet-only,
#                                                floor at break-even
#
#   BREAKEVEN_AT_R (Cell 4) optionally moves the stop to break-even as
#   soon as price travels that many R in favour, without waiting for
#   VWAP. It is OFF by default because it contradicts the playbook.
#   It exists because the first full run showed 23% of trades reaching
#   +1R and still finishing at -1R.
#
# ─────────────────────────────────────────────────────────────────────
# CASE 2 — VWAP is closer than 1R. Ladder suppressed entirely.
# ─────────────────────────────────────────────────────────────────────
#
#   VWAP REACHED    ->  sell 70%. NO break-even move.
#   remaining 30%   ->  trails, ratchet-only, floor at the ORIGINAL
#                       brake. The stop never improves to break-even
#                       in this branch.
#
# ─────────────────────────────────────────────────────────────────────
# WHAT A VOLUME EVENT IS
# ─────────────────────────────────────────────────────────────────────
#
# A bar, WHILE THE POSITION IS OPEN, whose volume makes it the new #1
# or #2 by volume in the session — the same pool the entry used. The
# signal candle itself is not counted; the event counter starts at zero
# on entry.
#
# EVENTS REQUIRE BEING IN PROFIT. The event bar must CLOSE beyond the
# entry price in the trade's favour. An adverse volume event does
# NOTHING — no cut, no stop change.
#
# This is deliberate and it is the heart of the strategy. In an Ending
# Volume trade, a monster bar against you near the low is thesis-
# CONFIRMING, not thesis-breaking. Cutting there is selling the bottom
# of your own setup. The brake already bounds the loss; nothing else
# needs to react to adverse volume.
#
# Volume events are PRE-VWAP ONLY. Once VWAP is reached the ladder is
# finished and the runner is governed by the trail alone.
#
# ─────────────────────────────────────────────────────────────────────
# THE VWAP ACTION
# ─────────────────────────────────────────────────────────────────────
#
# VWAP TOPS UP TO THE BRANCH TARGET AND NEVER SELLS BELOW WHAT IS
# ALREADY HELD. If Case 1 already sold 75% via two volume events, VWAP
# sells a further 15% to reach 90% — not another 90%.
#
# Reaching VWAP ARMS THE TRAIL whether or not anything was sold there.
#
# VWAP is the LIVE session value recomputed every bar. The value frozen
# at entry is used only to pick the branch.
#
# A trade counts as having reached VWAP on an INTRABAR TOUCH — the
# bar's high (long) or low (short) reaching the live VWAP.
#
# ─────────────────────────────────────────────────────────────────────
# THE TRAIL
# ─────────────────────────────────────────────────────────────────────
#
# The runner trails the ROLLING ~5-MINUTE LOW (long) / HIGH (short).
# Five minutes is exactly one bar on the 5-min. It does NOT divide into
# 2-min bars, so the 2-min rounds UP to three bars (6 minutes) rather
# than down to two (4 minutes): the ratchet means a slightly loose trail
# just moves less often, while a tight one cuts the runner that produces
# Case 1's whole tail. Set in TRAIL_BARS, Cell 4.
#
# RATCHET-ONLY: the stop moves only in the favourable direction and
# never comes back once moved.
#
# 52-week highs collapse into this trail. They are not a separate rule.
#
# ─────────────────────────────────────────────────────────────────────
# WITHIN-BAR PRECEDENCE
# ─────────────────────────────────────────────────────────────────────
#
# Bar data gives OHLCV but not the path inside the bar, so when several
# rules could fire in the same bar the order has to be assumed. This
# engine ASSUMES THE ADVERSE EVENT HAPPENED FIRST — losses are booked
# at the earliest plausible moment, gains at the latest. Where the
# truth is unknowable, the backtest understates.
#
#     1. hard brake / active trail   (intrabar; gap fills at the open)
#     2. VWAP action                 (intrabar touch)
#     3. volume event                (only if nothing above resolved)
#     4. soft stop                   (on the bar's CLOSE only)
#
# ─────────────────────────────────────────────────────────────────────
# SESSION RULES
# ─────────────────────────────────────────────────────────────────────
#
#   No new entries after 14:30. The closing-auction volume ramp makes
#   almost every late bar the session's new highest-volume candle, so
#   the rank rule fires constantly on what is just the clock. 44% of
#   all raw signals fire in the last hour and they are the only
#   negative bucket in the whole study.
#
#   Flat at 15:55, market, all remaining size, unconditionally.
#
#   DAILY LOSS LIMIT gates new entries once the day's realised result
#   breaches it. Positions already open run to their own exits.
#
#   Concurrency unlimited and re-entry unlimited on this first run —
#   we measure the raw edge, then add capacity constraints knowing
#   what the ceiling was.
#
#
# ═══════════════════════════════════════════════════════════════════════
# PART 4 — WHAT COMES OUT
# ═══════════════════════════════════════════════════════════════════════
#
#   ENDVOL_ladder_legs.csv     every fill: time, price, size, reason
#   ENDVOL_ladder_trades.csv   one row per trade: entry, branch, R,
#                              total R, MFE, MAE, exit reason, legs
#
# MFE and MAE are both recorded here. The penetration study only had
# MAE, which is why the dashboard has no efficiency ratio.
#
# COST is charged per SHARE, per side: once on entry over the full
# position, once on exit split across the ladder legs by size. A
# four-leg exit does NOT pay four times - each leg only pays for its
# own shares, so the totals come to exactly two sides either way.
# (An earlier draft of this notebook claimed otherwise. It was wrong:
# that is only true if your broker charges a fixed fee per ORDER, in
# which case raise COST_PER_SIDE to cover it.)
#
#
# ═══════════════════════════════════════════════════════════════════════
# PART 5 — VALIDATION GATE
# ═══════════════════════════════════════════════════════════════════════
#
# Cell 9 replays the three annotated trades from the playbook PDF —
# SNAP, NVDL 10:00/10:05, MSFU at $39 — and prints every leg so they
# can be checked by hand against the journal.
#
# DO NOT RUN THE FULL UNIVERSE UNTIL THOSE THREE RECONCILE.
# ═══════════════════════════════════════════════════════════════════════

print("Ending Volume — Ladder Backtest Engine")
print("Read Cell 1 before changing anything in Cell 4.")

In [ ]:
# ─── Cell 2: Imports ──────────────────────────────────────────────────
import pandas as pd                                    # dataframes for bars and results
import numpy as np                                     # vector maths on bar arrays
import os                                              # paths, cache dir, file checks
import datetime as dt                                  # time-of-day comparisons
from datetime import datetime                          # wall-clock for run timing
from dotenv import load_dotenv                         # read Alpaca keys from .env

from alpaca.data.historical import StockHistoricalDataClient   # bar fetch client
from alpaca.data.requests import StockBarsRequest              # bar request object
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit     # 2-min / 5-min specs
from alpaca.data.enums import DataFeed                         # SIP vs IEX selector

pd.set_option('display.width', 200)                    # wide console for leg tables
pd.set_option('display.max_columns', 50)               # don't truncate trade rows

print("✅ Libraries imported")

In [ ]:
# ─── Cell 3: Alpaca Connection ────────────────────────────────────────
BASE_PATH = (                                          # root of the AlgoTrading folder
    r"c:\Users\kwgit\Desktop\Playbook"
    r"\Claude_2025_2026_Playbooks"
    r"\AlgoTrading\AlgoTrading"
)

dotenv_path = BASE_PATH + r"\.env"                     # keys live beside the notebooks
load_dotenv(dotenv_path, override=True)                # override any stale shell vars

API_KEY    = os.getenv("ALPACA_API_KEY")               # public key
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")            # secret key

client = StockHistoricalDataClient(                    # market-data client only
    api_key    = API_KEY,                              # no trading client needed here
    secret_key = SECRET_KEY
)

print("✅ Alpaca connected")

In [ ]:
# ─── Cell 4: Configuration ────────────────────────────────────────────
#
# Every tunable lives here. Cell 1 explains what each one does to the
# strategy — read it before changing any of them.

WATCHLIST_FILE = BASE_PATH + r"\ENDVOL_watchlist.txt"          # tickers to scan
CACHE_DIR      = BASE_PATH + r"\ENDVOL_cache"                  # parquet bar cache
LEGS_OUT       = BASE_PATH + r"\ENDVOL_ladder_legs.csv"        # every individual fill
TRADES_OUT     = BASE_PATH + r"\ENDVOL_ladder_trades.csv"      # one row per trade

# ── Date range ────────────────────────────────────────────────────────
START_DATE = "2023-01-01"                              # study start
END_DATE   = "2026-09-02"                              # exclusive end. 09-02 so the
                                                       # SPCH 2026-09-01 trade in the
                                                       # playbook is inside the range

# ── Feed ──────────────────────────────────────────────────────────────
FEED = DataFeed.SIP                                    # full consolidated tape;
                                                       # IEX is ~2.5% of volume and
                                                       # would break every ranking

# ── Session windows (Eastern) ─────────────────────────────────────────
SESSION_OPEN  = dt.time(9, 30)                         # first bar of regular hours
SESSION_CLOSE = dt.time(16, 0)                         # end of regular hours
OR_END        = dt.time(9, 45)                         # 15-min opening range closes
RANK_START_2M = dt.time(9, 46)                         # rank path eligible (2-min)
RANK_START_5M = dt.time(9, 45)                         # rank path eligible (5-min)
X2_START_2M   = dt.time(9, 32)                         # 2x path eligible (2-min)
X2_START_5M   = dt.time(9, 35)                         # 2x path eligible (5-min)

# ── Entry cutoff and flatten ──────────────────────────────────────────
LAST_ENTRY    = dt.time(14, 30)                        # no new entries after this;
                                                       # the closing ramp makes the
                                                       # rank rule fire on the clock
FLATTEN_TIME  = dt.time(15, 55)                        # market-out everything here

# ── Signal criteria ───────────────────────────────────────────────────
TOP_RANK      = 2                                      # new #1 or #2 volume candle
X2_MULTIPLE   = 2.0                                    # or >= 2x reigning biggest
PRICE_FLOOR   = 4.00                                   # skip sub-$4 names
MIN_BAR_VOL   = 1_000                                  # ignore untradeably thin bars

# ── Stops ─────────────────────────────────────────────────────────────
BRAKE_FRAC     = 0.25                                  # brake sits this fraction of
                                                       # the signal candle's range
                                                       # beyond the soft stop
MIN_TARGET_R    = 0.25                                 # VWAP must be at least this
                                                       # many R away or there is nothing
                                                       # to aim at. TSLL 2026-08-25 10:30
                                                       # had VWAP 0.01R from entry — a
                                                       # round trip of cost for a target
                                                       # inside the spread.
MAX_COST_FRAC   = 0.05                                 # PRIMARY affordability gate:
                                                       # the round trip may not exceed
                                                       # this share of the distance to
                                                       # VWAP. Measured against the MOVE
                                                       # WE ARE AIMING FOR, not against R
                                                       # — a 4c risk unit chasing a 35c
                                                       # target is fine; a 4c risk unit
                                                       # chasing 6c is not. Self-adjusts
                                                       # with your real fills.
                                                       # Set to 0 to disable.
MIN_R_COST_MULT = 0.0                                  # secondary, OFF by default. A
                                                       # floor on R itself measured in
                                                       # round trips. At zero cost this
                                                       # gate strictly HURTS (small-R
                                                       # trades are good trades), so it
                                                       # exists only as a blunt backstop.

# ── Ladder ────────────────────────────────────────────────────────────
C1_EVENT1_CUT  = 0.50                                  # Case 1: first volume event
C1_EVENT2_CUT  = 0.25                                  # Case 1: second volume event
C1_VWAP_TARGET = 0.90                                  # Case 1: total sold at VWAP
C2_VWAP_TARGET = 0.70                                  # Case 2: total sold at VWAP
BREAKEVEN_AT_R = 0.0                                   # EXPERIMENTAL, off by default.
                                                       # When > 0, the stop moves to
                                                       # break-even as soon as price has
                                                       # travelled this many R in favour
                                                       # — WITHOUT waiting for VWAP.
                                                       # This CONTRADICTS the playbook,
                                                       # which moves to break-even only
                                                       # at VWAP. It exists because 23%
                                                       # of trades reached +1R and still
                                                       # finished at -1R: between entry
                                                       # and VWAP nothing protects a gain
                                                       # unless a volume event fires, and
                                                       # 92% of trades never get one.
                                                       # Set 0.5 / 0.75 / 1.0 to test.
                                                       # The cost of the rule — winners
                                                       # shaken out at break-even — is
                                                       # only visible by running it.
TRAIL_BARS     = {"2m": 3, "5m": 1}                    # bars in the rolling trail
                                                       # window. 5 minutes does not
                                                       # divide into 2-min bars, so
                                                       # the 2-min rounds UP to 3 bars
                                                       # (6 min): a trail 20% too loose
                                                       # only moves less often, while
                                                       # one 20% too tight cuts the
                                                       # runner that pays for Case 1

# ── Costs ─────────────────────────────────────────────────────────────
COST_PER_SIDE  = 0.0075                                # dollars per SHARE, per side.
                                                       # Charged once on entry over the
                                                       # full position and once on exit,
                                                       # split across the ladder legs by
                                                       # size. Splitting the exit into
                                                       # four partials does NOT cost four
                                                       # times - each leg only pays for
                                                       # its own shares.

# ── Daily loss limit ──────────────────────────────────────────────────
DAILY_LOSS_R   = 6.0                                   # stop opening new trades once
                                                       # the day is down this many R;
                                                       # set to None to disable
# ── Data quality guards ───────────────────────────────────────────────
MIN_OR_BARS_2M   = 6                                   # of 8 possible OR bars
MIN_OR_BARS_5M   = 3                                   # of 3 possible OR bars
MIN_SESSION_FRAC = 0.80                                # session must be this complete
FULL_BARS = {"2m": 195, "5m": 78}                      # expected bars per session

# ── Which timeframe to run ────────────────────────────────────────────
TIMEFRAMES = ["5m"]                                    # 5-min is primary for the
                                                       # playbook names; add "2m" to
                                                       # run both

# ── Segmentation ──────────────────────────────────────────────────────
PLAYBOOK_NAMES = {                                     # tag rows so index names never
    'SNAP','MRVL','DKNG','PLTU','SMCX','NVDL','SNXX','AMZU','MSFU',   # get pooled with
    'AAPU','TSLL','SPCH','SSPC','MRNA','SKHX','MVLL','IBIT','CONL',   # the 29 Kevin
    'AMDL','SNDQ','CMG','CRMG','AFRM','GME','DLLL','PL','IOT','HPE','MUU',  # trades
}

print("✅ Configuration loaded")
print(f"   Range:        {START_DATE} → {END_DATE}")
print(f"   Timeframes:   {', '.join(TIMEFRAMES)}")
print(f"   Brake:        {BRAKE_FRAC:.0%} of signal candle range")
MIN_R_DOLLARS = MIN_R_COST_MULT * 2 * COST_PER_SIDE    # derived, never hand-set
print(f"   Cost gate:    round trip <= {MAX_COST_FRAC:.0%} of the distance to VWAP"
      + ("   (DISABLED)" if MAX_COST_FRAC <= 0 else ""))
print(f"   Min R:        " + (f"${MIN_R_DOLLARS:.2f} = {MIN_R_COST_MULT:.0f} round trips"
                              if MIN_R_DOLLARS > 0 else "off (secondary gate)"))
print(f"   Cost:         ${COST_PER_SIDE:.4f}/share per leg")
print(f"   Trail window: " + ", ".join(f"{k} = {v} bar(s)" for k, v in TRAIL_BARS.items()))
print(f"   Min target:   {MIN_TARGET_R:.2f}R to VWAP  (closer than this is not a trade)")
print(f"   Break-even:   " + (f"at +{BREAKEVEN_AT_R:.2f}R  (EXPERIMENTAL, not the playbook rule)"
                              if BREAKEVEN_AT_R > 0 else "only at VWAP  (playbook rule)"))
print(f"   Daily limit:  {('-' + str(DAILY_LOSS_R) + 'R') if DAILY_LOSS_R else 'none'}")
print(f"   Last entry:   {LAST_ENTRY}   Flatten: {FLATTEN_TIME}")

In [ ]:
# ─── Cell 5: Universe Loader ──────────────────────────────────────────

def read_ticker_file(path, label):
    """Read one ticker file, skipping # comments and blank lines."""
    if not os.path.exists(path):                       # missing file is a warning,
        print(f"   ⚠️  {label}: not found at {path}")  # not a crash
        return []
    with open(path, 'r') as f:                         # plain text, one per line
        tickers = [
            line.strip().upper()                       # normalise case and whitespace
            for line in f
            if line.strip() and not line.strip().startswith('#')   # drop comments
        ]
    print(f"   {label:<10} {len(tickers):>4} tickers")
    return tickers


universe = read_ticker_file(WATCHLIST_FILE, "ENDVOL")             # read the watchlist
universe = sorted(set(t for t in universe if '.' not in t))       # dedupe; Alpaca
                                                                  # chokes on BRK.B
os.makedirs(CACHE_DIR, exist_ok=True)                             # cache dir must exist

print(f"\n   ✅ Universe: {len(universe)} unique tickers")
print(f"   ✅ Cache dir: {CACHE_DIR}")

In [ ]:
# ─── Cell 6: Bar Fetch + Cache ────────────────────────────────────────
#
# Identical to the penetration study, so the parquet cache it already
# built is reused as-is. Nothing re-downloads unless force=True.

TF = {
    "2m": TimeFrame(2, TimeFrameUnit.Minute),          # 2-minute bars
    "5m": TimeFrame(5, TimeFrameUnit.Minute),          # 5-minute bars
}


def cache_path(symbol, tf_label):
    """Parquet path for one ticker/timeframe."""
    return os.path.join(CACHE_DIR, f"{symbol}_{tf_label}.parquet")


def fetch_bars(symbol, tf_label, force=False):
    """Regular-hours bars for one ticker/timeframe, cached to parquet."""
    path = cache_path(symbol, tf_label)                # where this ticker's bars live
    if os.path.exists(path) and not force:             # cache hit — no network call
        return pd.read_parquet(path)

    req = StockBarsRequest(                            # build the Alpaca request
        symbol_or_symbols = symbol,
        timeframe         = TF[tf_label],
        start             = START_DATE,
        end               = END_DATE,
        feed              = FEED,                      # SIP, per Cell 4
    )
    df = client.get_stock_bars(req).df                 # returns a MultiIndex frame

    if df is None or len(df) == 0:                     # delisted / no data
        empty = pd.DataFrame(columns=['open','high','low','close','volume'])
        empty.to_parquet(path)                         # cache the emptiness too
        return empty

    if isinstance(df.index, pd.MultiIndex):            # drop the symbol level
        df = df.xs(symbol, level=0)

    df = df[['open','high','low','close','volume']].copy()   # keep only OHLCV
    df.index = pd.to_datetime(df.index, utc=True)            # normalise to UTC first
    df.index = df.index.tz_convert('America/New_York')       # then to Eastern

    df = df[(df.index.time >= SESSION_OPEN) &          # regular hours only; pre/post
            (df.index.time <  SESSION_CLOSE)]          # volume would break rankings

    df.to_parquet(path)                                # persist for next run
    return df


print("✅ Bar fetch ready (cache reused from the penetration study)")

In [ ]:
# ─── Cell 7: Signal Detection ─────────────────────────────────────────
#
# Unchanged from the penetration study so the two runs describe the
# same population. See Cell 1, Part 1 for the rules in prose.


def session_vwap(day_df):
    """Session VWAP from 9:30, typical-price weighted."""
    tp = (day_df['high'] + day_df['low'] + day_df['close']) / 3.0   # typical price
    cum_pv  = (tp * day_df['volume']).cumsum()         # running price x volume
    cum_vol = day_df['volume'].cumsum()                # running volume
    return cum_pv / cum_vol.replace(0, np.nan)         # guard divide-by-zero


def find_signals(day_df, tf_label, vwap):
    """Return (signals, skip_reason) for one ticker-session."""
    min_or = MIN_OR_BARS_2M if tf_label == '2m' else MIN_OR_BARS_5M   # coverage bar

    if len(day_df) < FULL_BARS[tf_label] * MIN_SESSION_FRAC:   # too many missing bars
        return [], 'sparse_session'

    rank_start = RANK_START_2M if tf_label == '2m' else RANK_START_5M  # rank eligible
    x2_start   = X2_START_2M   if tf_label == '2m' else X2_START_5M    # 2x eligible

    or_mask = day_df.index.time < OR_END               # the 9:30-9:45 window
    if or_mask.sum() < min_or:                         # range would be fiction
        return [], 'thin_opening_range'

    or_high = day_df.loc[or_mask, 'high'].max()        # top of the opening range
    or_low  = day_df.loc[or_mask, 'low'].min()         # bottom of the opening range
    if not (or_high > or_low):                         # degenerate single-print range
        return [], 'degenerate_range'

    sess_open = day_df['open'].iloc[0]                 # first print of the session

    vols  = day_df['volume'].to_numpy()                # arrays are much faster than
    highs = day_df['high'].to_numpy()                  # .iloc in the inner loop
    lows  = day_df['low'].to_numpy()
    times = day_df.index

    signals = []

    for i in range(1, len(day_df)):                    # bar 0 has no prior to rank against
        v = vols[i]                                    # this bar's volume
        t = times[i].time()                            # this bar's clock time
        if v < MIN_BAR_VOL:                            # untradeably thin
            continue

        prior_max = vols[:i].max()                     # reigning biggest so far
        rank      = int((vols[:i+1] > v).sum()) + 1    # 1 = biggest of the session

        is_x2   = (t >= x2_start) and prior_max > 0 and v >= X2_MULTIPLE * prior_max
        is_rank = (t >= rank_start) and rank <= TOP_RANK

        if not (is_x2 or is_rank):                     # neither path fired
            continue

        after_or = t >= OR_END                         # is the range complete yet
        if after_or:
            below = lows[i]  < or_low                  # poked under the range
            above = highs[i] > or_high                 # poked over the range
            if below and above:                        # engulfed the whole range —
                direction = ('long' if (or_low - lows[i]) >= (highs[i] - or_high)
                             else 'short')             # take the further excursion
            elif below:
                direction = 'long'                     # counter-trend: fade the break
            elif above:
                direction = 'short'
            else:
                continue                               # inside the range — no trade
        else:
            if lows[i] <= lows[:i+1].min():            # new low of day
                direction = 'long'
            elif highs[i] >= highs[:i+1].max():        # new high of day
                direction = 'short'
            else:
                direction = ('long' if day_df['close'].iloc[i] < sess_open
                             else 'short')             # fall back to vs session open

        signals.append({
            'bar_idx'    : i,                          # index into day_df
            'time'       : times[i],                   # signal candle timestamp
            'tf'         : tf_label,
            'direction'  : direction,
            'path'       : '2x' if is_x2 else 'rank',
            'rank'       : rank,
            'volume'     : v,
            'high'       : highs[i],
            'low'        : lows[i],
            'close'      : day_df['close'].iloc[i],
            'candle_rng' : highs[i] - lows[i],         # the brake's unit of measure
            'vwap_at_sig': vwap.iloc[i],               # frozen value for the branch gate
            'or_high'    : or_high,
            'or_low'     : or_low,
            'pre_or'     : not after_or,
        })

    return signals, None


print("✅ Signal detection ready")

In [ ]:
# ─── Cell 8: The Ladder Engine ────────────────────────────────────────
#
# Walks ONE trade forward bar by bar and returns every fill it made.
# Exit rules are spelled out in Cell 1, Part 3. Within-bar precedence
# is: brake/trail -> VWAP -> volume event -> soft stop on close.


def simulate_trade(day_df, vwap, sig, tf_label):
    """Simulate one signal end to end. Returns (trade_dict, legs) or (None, reason)."""

    i0 = sig['bar_idx']                                # index of the signal candle
    if i0 + 1 >= len(day_df):                          # signal on the last bar —
        return None, 'no_entry_bar'                    # nothing to enter on

    if sig['time'].time() >= LAST_ENTRY:               # past the entry cutoff
        return None, 'after_cutoff'

    ei       = i0 + 1                                  # entry bar index
    entry_px = float(day_df['open'].iloc[ei])          # fill at the next bar's open
    entry_t  = day_df.index[ei]                        # entry timestamp
    long     = sig['direction'] == 'long'              # direction flag, used everywhere
    sgn      = 1.0 if long else -1.0                   # +1 long, -1 short

    if entry_px < PRICE_FLOOR:                         # sub-$4 names excluded
        return None, 'below_price_floor'

    rng  = float(sig['candle_rng'])                    # signal candle high minus low
    if rng <= 0:
        return None, 'degenerate_candle'

    soft  = float(sig['low']) if long else float(sig['high'])   # soft stop level
    brake = soft - BRAKE_FRAC * rng if long else soft + BRAKE_FRAC * rng   # hard brake
    if (long and entry_px <= brake) or ((not long) and entry_px >= brake):
        return None, 'gap_past_brake'                  # entry bar opened through the
                                                       # brake — stopped before we
                                                       # started; abs() would hide it

    if (long and entry_px <= soft) or ((not long) and entry_px >= soft):
        return None, 'entry_past_soft_stop'            # the next bar opened BEYOND the
                                                       # signal candle's extreme, so we
                                                       # would be entering on the wrong
                                                       # side of our own soft stop and
                                                       # the remaining R is whatever
                                                       # slice of the brake buffer is
                                                       # left. MSFU 2026-08-27 did this:
                                                       # short at 39.070 with the stop
                                                       # at 39.055, leaving R = $0.006
                                                       # and a 'brake' that cost 2.2R.

    R = abs(entry_px - brake)                          # THE risk unit, entry to brake

    if R < MIN_R_DOLLARS:                              # too tight to trade after costs
        return None, 'below_min_R'

    vwap_e = float(sig['vwap_at_sig'])                 # VWAP frozen at the signal close
    target_dist = (vwap_e - entry_px) * sgn            # distance to target, signed
    if target_dist <= 0:                               # VWAP already behind us
        return None, 'target_behind'

    if target_dist < MIN_TARGET_R * R:                 # target too close to bother
        return None, 'target_too_close'

    if MAX_COST_FRAC > 0 and (2 * COST_PER_SIDE) > MAX_COST_FRAC * target_dist:
        return None, 'cost_vs_target'                  # friction is too large a share of
                                                       # the move we are aiming for. This
                                                       # is the PRIMARY affordability
                                                       # test — see Cell 1.

    branch = 'CASE1' if target_dist >= R else 'CASE2'  # THE gate — fixed for the trade
    vwap_target = C1_VWAP_TARGET if branch == 'CASE1' else C2_VWAP_TARGET

    # ── running state ─────────────────────────────────────────────────
    sold        = 0.0                                  # fraction of original sold
    vol_events  = 0                                    # in-profit volume events used
    vwap_done   = False                                # has the VWAP action fired
    trail       = None                                 # active trailing stop level
    stop_floor  = brake                                # worst level the stop may take
    legs        = []                                   # every fill we make
    mfe         = 0.0                                  # max favourable excursion, $
    mae         = 0.0                                  # max adverse excursion, $
    exit_reason = None

    vols  = day_df['volume'].to_numpy()                # for volume-event ranking
    lows  = day_df['low'].to_numpy()
    highs = day_df['high'].to_numpy()

    trail_bars = TRAIL_BARS[tf_label]                  # explicit, see Cell 4

    def book(idx, px, frac, reason):
        """Record one fill and add it to the sold total."""
        nonlocal sold
        if frac <= 1e-9:                               # nothing to sell
            return
        legs.append({
            'time'   : day_df.index[idx],              # when it filled
            'price'  : float(px),                      # where it filled
            'frac'   : float(frac),                    # fraction of ORIGINAL position
            'reason' : reason,                         # why it fired
            'R_leg'  : (float((px - entry_px) * sgn) - COST_PER_SIDE) * frac / R,
        })                                             # this leg's exit cost only;
                                                       # entry is charged once below
        sold += frac

    # ── walk forward from the entry bar ───────────────────────────────
    for i in range(ei, len(day_df)):
        o = float(day_df['open'].iloc[i])              # this bar's open
        h = float(day_df['high'].iloc[i])              # high
        l = float(day_df['low'].iloc[i])               # low
        c = float(day_df['close'].iloc[i])             # close
        t = day_df.index[i].time()                     # clock time of this bar
        vw_now = float(vwap.iloc[i])                   # LIVE vwap this bar

        mfe = max(mfe, (h - entry_px) if long else (entry_px - l))   # best seen, $
        mae = max(mae, (entry_px - l) if long else (h - entry_px))   # worst seen, $

        # ── 1. hard brake or active trail, intrabar, always ───────────
        stop_now = trail if trail is not None else brake        # whichever governs
        hit = (l <= stop_now) if long else (h >= stop_now)      # did price reach it
        if hit:
            gap = (o < stop_now) if long else (o > stop_now)    # opened through it
            fill = o if gap else stop_now                       # gaps fill at the open
            book(i, fill, 1.0 - sold, 'trail' if trail is not None else 'brake')
            exit_reason = ('gap_' if gap else '') + ('trail' if trail is not None else 'brake')
            break

        # ── 2. flatten at the bell ────────────────────────────────────
        if t >= FLATTEN_TIME:
            book(i, c, 1.0 - sold, 'flatten_1555')     # market out, no exceptions
            exit_reason = 'flatten_1555'
            break

        # ── 3. VWAP action, intrabar touch of the LIVE vwap ───────────
        if not vwap_done:
            touched = (h >= vw_now) if long else (l <= vw_now)  # did we tag it
            if touched:
                book(i, vw_now, max(0.0, vwap_target - sold), 'vwap')  # top up only
                vwap_done = True
                if branch == 'CASE1':
                    stop_floor = entry_px              # Case 1 moves to break-even
                trail = stop_floor                     # arm the trail at the floor
                if sold >= 1.0 - 1e-9:                 # nothing left to run
                    exit_reason = 'vwap_full'
                    break

        # ── 3b. early break-even, if enabled (NOT the playbook rule) ──
        if BREAKEVEN_AT_R > 0 and not vwap_done:
            excursion = (h - entry_px) if long else (entry_px - l)   # best this bar
            if excursion >= BREAKEVEN_AT_R * R:         # travelled far enough
                stop_floor = entry_px                   # never worse than flat
                trail = (max(trail, stop_floor) if long else min(trail, stop_floor)) \
                        if trail is not None else stop_floor        # ratchet only

        # ── 4. volume event — pre-VWAP only, must be in profit ────────
        if (not vwap_done) and branch == 'CASE1' and vol_events < 2:
            rank_i = int((vols[:i+1] > vols[i]).sum()) + 1      # rank within session
            if rank_i <= TOP_RANK:                              # new #1 or #2
                in_profit = (c > entry_px) if long else (c < entry_px)   # CLOSE test
                if in_profit:                                   # adverse events do nothing
                    vol_events += 1
                    cut = C1_EVENT1_CUT if vol_events == 1 else C1_EVENT2_CUT
                    book(i, c, cut, f'vol_event_{vol_events}')  # fill at the close

        # ── 5. soft stop — CLOSES ONLY, and only before any partial ───
        if sold <= 1e-9:                               # retired once the trade has paid
            closed_through = (c < soft) if long else (c > soft)
            if closed_through:
                book(i, c, 1.0, 'soft_stop')
                exit_reason = 'soft_stop'
                break

        # ── 6. ratchet the trail once armed ───────────────────────────
        if trail is not None and i >= ei + trail_bars:
            lo_win = lows[i - trail_bars + 1 : i + 1].min()  # rolling window low
            hi_win = highs[i - trail_bars + 1 : i + 1].max() # rolling window high
            cand   = lo_win if long else hi_win                 # candidate stop
            cand   = max(cand, stop_floor) if long else min(cand, stop_floor)  # floor it
            trail  = max(trail, cand) if long else min(trail, cand)  # ratchet only

    if sold < 1.0 - 1e-9:                              # ran out of bars (shouldn't happen)
        book(len(day_df) - 1, float(day_df['close'].iloc[-1]), 1.0 - sold, 'eod')
        exit_reason = exit_reason or 'eod'

    total_R = sum(lg['R_leg'] for lg in legs) - (COST_PER_SIDE / R)  # entry cost once

    trade = {
        'symbol'      : None,                          # filled in by the caller
        'session'     : str(day_df.index[0].date()),
        'tf'          : tf_label,
        'signal_time' : sig['time'],
        'entry_time'  : entry_t,
        'entry_px'    : entry_px,
        'direction'   : sig['direction'],
        'path'        : sig['path'],
        'branch'      : branch,
        'soft_stop'   : soft,
        'brake'       : brake,
        'R_dollar'    : R,
        'vwap_at_entry': vwap_e,
        'target_R'    : target_dist / R,               # how far VWAP was, in R
        'vol_events'  : vol_events,
        'reached_vwap': vwap_done,
        'n_legs'      : len(legs),
        'exit_reason' : exit_reason,
        'mfe_R'       : mfe / R,                       # best it ever showed
        'mae_R'       : mae / R,                       # worst it ever showed
        'total_R'     : total_R,                       # the number that matters
    }
    return trade, legs


print("✅ Ladder engine ready")

In [ ]:
# ─── Cell 9: VALIDATION GATE — the three annotated playbook trades ────
#
# Replays SNAP, NVDL and MSFU and prints every leg so they can be
# checked by hand against the journal. Nothing scales until these
# reconcile with what the PDF records.
#
# Set the dates to the sessions your playbook annotates.

# Dates read off the chart headers in the playbook PDF.
VALIDATE = [
    ("MSFU", "2026-08-27", "5m"),    # 13:40 short, HOD retest ~$39.05   R = 2.6c
    ("NVDL", "2026-08-28", "5m"),    # 10:05 long, the TRUE HIGHEST seq   R = 13.0c
    ("TSLL", "2026-08-25", "5m"),    # 10:10 short                        R = 20.2c
    ("SPCH", "2026-08-26", "5m"),    # 09:55 short at $8.85               R = 13.8c
    ("NVDL", "2026-08-26", "2m"),    # 13:12 long, 2x the open            R = 2.8c
]

# Four of those five sit UNDER the production floor, so validation runs
# with it disabled - otherwise the engine prints "below_min_R" five times
# and tells us nothing about the ladder. Cell 10 restores it.
_SAVED_MIN_R, _SAVED_FRAC = MIN_R_DOLLARS, MAX_COST_FRAC
MIN_R_DOLLARS, MAX_COST_FRAC = 0.0, 0.0
print(f"⚠️  Affordability gates disabled for validation "
      f"(cost<={_SAVED_FRAC:.0%} of target, restored in Cell 10)\n")


def replay(symbol, session, tf_label="5m"):
    """Print every signal and every fill for one ticker-session."""
    bars = fetch_bars(symbol, tf_label)                # cached bars
    if len(bars) == 0:
        print(f"   {symbol} {session}: no bars"); return
    day = bars[bars.index.normalize() == pd.Timestamp(session, tz='America/New_York')]
    if len(day) == 0:
        print(f"   {symbol} {session}: no session"); return

    vw = session_vwap(day)                             # VWAP for this session
    sigs, skip = find_signals(day, tf_label, vw)       # every qualifying candle
    if skip:
        print(f"   {symbol} {session}: session skipped ({skip})"); return

    print(f"\n{'='*74}\n{symbol}  {session}  [{tf_label}]   "
          f"OR {sigs[0]['or_low']:.2f}-{sigs[0]['or_high']:.2f}" if sigs else
          f"\n{symbol} {session}: no signals")
    if not sigs:
        return
    print('='*74)

    for s in sigs:
        tr, legs = simulate_trade(day, vw, s, tf_label)
        stamp = s['time'].strftime('%H:%M')
        if tr is None:
            print(f"  {stamp}  {s['direction']:<5} {s['path']:<4} "
                  f"vol {s['volume']:>10,.0f}   SKIPPED: {legs}")
            continue
        print(f"\n  {stamp}  {s['direction'].upper():<5} {s['path']:<4} "
              f"rank {s['rank']}  vol {s['volume']:>10,.0f}")
        print(f"      entry {tr['entry_px']:.3f}  soft {tr['soft_stop']:.3f}  "
              f"brake {tr['brake']:.3f}  R ${tr['R_dollar']:.3f}")
        print(f"      VWAP {tr['vwap_at_entry']:.3f} = {tr['target_R']:.2f}R away"
              f"   ->  {tr['branch']}")
        for lg in legs:
            print(f"        {lg['time'].strftime('%H:%M')}  "
                  f"{lg['reason']:<14} {lg['frac']*100:>5.1f}%  "
                  f"@ {lg['price']:.3f}   {lg['R_leg']:+.3f}R")
        print(f"      TOTAL {tr['total_R']:+.3f}R   "
              f"MFE {tr['mfe_R']:.2f}R  MAE {tr['mae_R']:.2f}R   "
              f"exit: {tr['exit_reason']}")


for sym, ses, tf_label in VALIDATE:
    replay(sym, ses, tf_label)

print(f"\n{'='*74}")
print("Check these legs against the playbook by hand before running Cell 10.")

In [ ]:
# ─── Cell 10: Run the Backtest ────────────────────────────────────────
#
# Walks the universe. Within each session, signals are processed in
# time order so the daily loss limit can gate later entries.

def run_ticker(symbol):
    """All trades for one ticker across the study range."""
    out_trades, out_legs, skips = [], [], {}
    in_pb = symbol in PLAYBOOK_NAMES                   # segmentation tag

    for tf_label in TIMEFRAMES:
        bars = fetch_bars(symbol, tf_label)            # cached parquet
        if len(bars) == 0:
            skips['no_bars'] = skips.get('no_bars', 0) + 1
            continue

        for day_ts, day in bars.groupby(bars.index.normalize()):   # session by session
            vw = session_vwap(day)                     # this session's VWAP
            sigs, skip = find_signals(day, tf_label, vw)
            if skip:
                skips[skip] = skips.get(skip, 0) + 1   # tally why it was dropped
                continue
            skips['studied'] = skips.get('studied', 0) + 1

            day_R = 0.0                                # realised R so far today
            halted = False                             # has the daily limit tripped

            for s in sorted(sigs, key=lambda x: x['bar_idx']):     # strict time order
                if halted:                             # limit hit — no new entries
                    skips['daily_limit'] = skips.get('daily_limit', 0) + 1
                    continue

                tr, legs = simulate_trade(day, vw, s, tf_label)
                if tr is None:
                    skips[legs] = skips.get(legs, 0) + 1           # legs holds the reason
                    continue

                tr['symbol'] = symbol                  # stamp identity
                tr['in_playbook'] = in_pb
                out_trades.append(tr)
                for lg in legs:                        # flatten legs for the CSV
                    lg2 = dict(lg)
                    lg2['symbol']      = symbol
                    lg2['session']     = tr['session']
                    lg2['entry_time']  = tr['entry_time']
                    lg2['branch']      = tr['branch']
                    out_legs.append(lg2)

                day_R += tr['total_R']                 # accumulate the day
                if DAILY_LOSS_R and day_R <= -DAILY_LOSS_R:        # breach the limit
                    halted = True                      # open trades still ran to exit

    return out_trades, out_legs, skips


MIN_R_DOLLARS = MIN_R_COST_MULT * 2 * COST_PER_SIDE     # restore after Cell 9
MAX_COST_FRAC = 0.05                                    # primary gate back on
print(f"Gates in force: round trip <= {MAX_COST_FRAC:.0%} of target"
      f"{'' if MIN_R_DOLLARS<=0 else f', min R ${MIN_R_DOLLARS:.2f}'}\n")

all_skips, n_tr, n_lg = {}, 0, 0
wrote_t = wrote_l = False
t0 = datetime.now()

for f in (TRADES_OUT, LEGS_OUT):                       # start clean
    if os.path.exists(f):
        os.remove(f)

for n, sym in enumerate(universe, 1):                  # one ticker at a time
    trades, legs, skips = run_ticker(sym)
    if trades:
        pd.DataFrame(trades).to_csv(TRADES_OUT, mode='a', header=not wrote_t, index=False)
        wrote_t = True
    if legs:
        pd.DataFrame(legs).to_csv(LEGS_OUT, mode='a', header=not wrote_l, index=False)
        wrote_l = True
    n_tr += len(trades); n_lg += len(legs)
    for k, v in skips.items():
        all_skips[k] = all_skips.get(k, 0) + v
    el  = (datetime.now() - t0).total_seconds()        # elapsed
    eta = el / n * (len(universe) - n) / 60            # projected minutes remaining
    print(f"   [{n:>3}/{len(universe)}] {sym:<6} {len(trades):>5} trades  "
          f"(total {n_tr:,})   ETA {eta:>5.0f} min", flush=True)

print(f"\n✅ Complete in {(datetime.now()-t0).total_seconds()/60:.1f} min")
print(f"   Trades: {n_tr:,}    Legs: {n_lg:,}")
print(f"   → {TRADES_OUT}")
print(f"   → {LEGS_OUT}")
print(f"\n   Accounting:")
for k, v in sorted(all_skips.items(), key=lambda kv: -kv[1]):
    print(f"      {k:<24} {v:>9,}")

In [ ]:
# ─── Cell 11: Results ─────────────────────────────────────────────────

T = pd.read_csv(TRADES_OUT) if os.path.exists(TRADES_OUT) else pd.DataFrame()
if len(T) == 0:
    print("No trades. Check Cell 9 first.")
else:
    def block(x, label):
        """One summary line for a slice of trades."""
        if len(x) == 0:
            return
        w = (x.total_R > 0)                            # a win is positive R, not
        print(f"  {label:<26}{len(x):>8,}{w.mean()*100:>8.1f}%"   # 'reached VWAP' —
              f"{x.total_R.mean():>+9.3f}{x.total_R.sum():>10,.0f}"  # the ladder makes
              f"{x.mfe_R.median():>8.2f}{x.mae_R.median():>8.2f}")   # partial wins real

    print(f"  {'slice':<26}{'trades':>8}{'win%':>8}{'expR':>9}{'totR':>10}"
          f"{'MFE':>8}{'MAE':>8}")
    print("  " + "-"*77)
    block(T, "ALL")
    for pb in (True, False):
        block(T[T.in_playbook == pb], "playbook" if pb else "index")
    print()
    for br in ('CASE1', 'CASE2'):
        block(T[T.branch == br], br)
    print()
    for r in sorted(T.exit_reason.dropna().unique()):
        block(T[T.exit_reason == r], f"exit: {r}")
    print()
    for ve in sorted(T.vol_events.unique()):
        block(T[T.vol_events == ve], f"{ve} volume event(s)")

    print(f"\n  Ladder usage:")
    print(f"     trades with >=1 volume event   {(T.vol_events>0).mean()*100:>6.1f}%")
    print(f"     trades reaching VWAP           {T.reached_vwap.mean()*100:>6.1f}%")
    print(f"     median legs per trade          {T.n_legs.median():>6.0f}")
    print(f"\n  Efficiency (now that MFE exists):")
    print(f"     median MFE                     {T.mfe_R.median():>6.2f}R")
    print(f"     median captured / MFE          "
          f"{(T.total_R/T.mfe_R.replace(0,np.nan)).median():>6.2f}")

    daily = T.groupby('session').total_R.sum().sort_index()   # daily equity path
    cum   = daily.cumsum()
    dd    = cum - cum.cummax()
    print(f"\n  Path:")
    print(f"     final                          {cum.iloc[-1]:>+8,.0f}R")
    print(f"     max drawdown                   {dd.min():>+8,.0f}R")
    print(f"     worst day                      {daily.min():>+8.1f}R")
    print(f"     losing days                    {(daily<0).mean()*100:>7.1f}%")